# Exercise 4 — build_agent_step_prompt and parse_agent_action

The agent decision layer: at each step the agent sees the question plus accumulated context and decides whether to retrieve more or answer.  `build_agent_step_prompt` constructs that decision prompt.  `parse_agent_action` parses the LLM's JSON response, falling back to a retrieve action on any parse failure.

In [ ]:
import json
from dataclasses import dataclass, field
@dataclass
class Document:
    content: str
    metadata: dict = field(default_factory=dict)

class SimpleRetriever:
    def __init__(self):
        self._docs = []
    def add(self, doc):
        self._docs.append(doc); return self
    def add_all(self, docs):
        for d in docs: self._docs.append(d)
        return self
    def _score(self, query, doc):
        q = set(query.lower().split())
        d = set(doc.content.lower().split())
        return len(q & d) / (len(q | d) + 1e-9)
    def search(self, query, top_k=3):
        if not self._docs: return []
        return sorted(self._docs, key=lambda doc: self._score(query, doc), reverse=True)[:top_k]
    def __len__(self): return len(self._docs)
def call_llm(messages, llm_fn=None):
    if llm_fn is not None:
        return str(llm_fn(messages))
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]

def safe_parse_json(text):
    start = str(text).find("{")
    end   = str(text).rfind("}") + 1
    if start == -1 or end == 0:
        return None
    try:
        return json.loads(text[start:end])
    except (json.JSONDecodeError, ValueError):
        return None
def format_docs(docs):
    if not docs:
        return "No documents found."
    lines = []
    for i, d in enumerate(docs, 1):
        source = d.metadata.get("source", f"doc{i}")
        lines.append(f"[{i}] ({source}) {d.content}")
    return "\n".join(lines)

def build_retrieval_prompt(question, docs):
    context = format_docs(docs)
    system = "\n".join([
        "You are a helpful assistant.",
        "Answer the question using ONLY the provided documents.",
        "If the answer is not in the documents, say: I don't have enough information.",
        "Cite document numbers like [1] when referencing specific facts.",
    ])
    user = "Documents:\n" + context + "\n\nQuestion: " + str(question)
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

def retrieve_and_answer(question, retriever, top_k=3, llm_fn=None):
    docs = retriever.search(question, top_k=top_k)
    prompt = build_retrieval_prompt(question, docs)
    answer = call_llm(prompt, llm_fn=llm_fn)
    return {"question": question, "docs": docs, "answer": answer}

# ── Exercise: implement build_agent_step_prompt and parse_agent_action ───────

def build_agent_step_prompt(question, context):
    # TODO: check if context is non-empty and not "No documents found."
    # Build a system message explaining the two JSON actions:
    #   {"action": "retrieve", "query": "..."} and {"action": "answer", "text": "..."}
    # Build user message: "Question: " + question + "\n\n" + context summary
    # Return [{"role": "system", ...}, {"role": "user", ...}]
    return [{"role": "system", "content": ""}, {"role": "user", "content": str(question)}]


def parse_agent_action(text):
    # TODO: call safe_parse_json(text) -> data dict
    # If data.get("action") == "answer": return {"action": "answer", "text": str(data.get("text", ""))}
    # Otherwise: return {"action": "retrieve", "query": str(data.get("query", ""))}
    # Never raise — fall back to retrieve on bad parse
    return {"action": "retrieve", "query": ""}


### Checks

In [ ]:
import json
checks = 0

# 1 — build_agent_step_prompt returns [system, user]
try:
    prompt = build_agent_step_prompt("What is AI?", "No context retrieved yet.")
    assert isinstance(prompt, list) and len(prompt) == 2
    assert prompt[0]["role"] == "system" and prompt[1]["role"] == "user"
    checks += 1; print("✅ 1 build_agent_step_prompt returns [system, user]")
except Exception as e:
    print("❌ 1:", e)

# 2 — prompt mentions both actions
try:
    prompt = build_agent_step_prompt("What is AI?", "No context retrieved yet.")
    sys_content = prompt[0]["content"]
    assert "retrieve" in sys_content and "answer" in sys_content
    checks += 1; print("✅ 2 system message mentions both retrieve and answer actions")
except Exception as e:
    print("❌ 2:", e)

# 3 — parse_agent_action: answer action
try:
    text = json.dumps({"action": "answer", "text": "AI is artificial intelligence."})
    act = parse_agent_action(text)
    assert act["action"] == "answer"
    assert act["text"] == "AI is artificial intelligence."
    checks += 1; print("✅ 3 parse_agent_action returns answer action correctly")
except Exception as e:
    print("❌ 3:", e)

# 4 — parse_agent_action: retrieve action
try:
    text = json.dumps({"action": "retrieve", "query": "artificial intelligence history"})
    act = parse_agent_action(text)
    assert act["action"] == "retrieve"
    assert "artificial" in act["query"]
    checks += 1; print("✅ 4 parse_agent_action returns retrieve action correctly")
except Exception as e:
    print("❌ 4:", e)

# 5 — parse_agent_action falls back to retrieve on bad JSON
try:
    act = parse_agent_action("this is not json at all")
    assert act["action"] == "retrieve"
    checks += 1; print("✅ 5 parse_agent_action falls back to retrieve on bad input")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
